# **ginger**

In [17]:
import sys
import os

if "google.colab" in sys.modules:
    import getpass

    token = getpass.getpass()

    repo_url = f"https://{token}@github.com/Mango-Cats/ginger.git"

    if os.path.exists("ginger"):
        %cd ginger
        !git pull origin main
    else:
        !git clone {repo_url}
        %cd ginger

    try:
        %load_ext autoreload
        %autoreload 2
    except Exception as e:
        print("autoreload not available:", e)

    !pip install -e . --no-deps

Cloning into 'ginger'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 64 (delta 25), reused 50 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 239.79 KiB | 18.44 MiB/s, done.
Resolving deltas: 100% (25/25), done.
/content/ginger/ginger/ginger
autoreload not available: No module named 'imp'
Obtaining file:///content/ginger/ginger/ginger
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ginger (pyproject.toml) ... done
  Created wheel for ginger: filename=ginger-0.1.0-0.editable-py3-none-any.whl size=5478 sha256=d81cfd128e4eebe8b3e2240b6cd331bf0394ace354028bcaa1688facf6fe61de
  Stored in directory: /tmp/pip-ephem-wheel-cache-vmhdg4ln/wheels/85/43/f2/82cdd52c8ab20cc2c99066e7761c2724754e712

In [18]:
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__)
print(DEVICE)

2.10.0+cu128
cuda


## **Data Loading & Preprocessing**

In [19]:
from src.dataman import create_dataloaders
from src.dataman.constants import PAD_IDX
from pathlib import Path

In [20]:
DATA_PATH = Path("data/wikipron_tagalog.tsv")
BATCH_SIZE = 64

train_loader, val_loader, test_loader, grapheme_vocab, phoneme_vocab = create_dataloaders(
    DATA_PATH,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

print(f"Train samples: {len(train_loader.dataset)}")
print(f"Val samples: {len(val_loader.dataset)}")
print(f"Test samples: {len(test_loader.dataset)}")
print(f"Grapheme vocab size: {len(grapheme_vocab)}")
print(f"Phoneme vocab size: {len(phoneme_vocab)}")

Train samples: 14590
Val samples: 1824
Test samples: 1824
Grapheme vocab size: 32
Phoneme vocab size: 34


## **Model Initialization**

In [21]:
from src.architecture.transformer import Transformer
from torch.optim.lr_scheduler import OneCycleLR
from torch.optim import AdamW
import torch.nn as nn

In [22]:
EMBEDDING_DIM = 128
NUM_ATTENTION_HEADS = 4
NUM_ENCODER_LAYERS = 4
NUM_DECODER_LAYERS = 4
FEEDFORWARD_DIM = 512
DROPOUT_RATE = 0.1
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 100 

model = Transformer(
    num_graphemes=len(grapheme_vocab),
    num_phonemes=len(phoneme_vocab),
    embedding_dim=EMBEDDING_DIM,
    num_attention_heads=NUM_ATTENTION_HEADS,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    feedforward_dim=FEEDFORWARD_DIM,
    dropout_rate=DROPOUT_RATE,
).to(DEVICE)

model = torch.compile(model)

CRITERION = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
OPTIMIZER = AdamW(params=model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
SCHEDULER = OneCycleLR(
    optimizer=OPTIMIZER,
    max_lr=LEARNING_RATE,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS,
    pct_start=0.1 
)
SCALER = torch.cuda.amp.GradScaler()
PATIENCE = 7

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(
/tmp/ipykernel_16149/2010783063.py:33: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  SCALER = torch.cuda.amp.GradScaler()


## **Training**

In [23]:
from src.trainer import evaluate, train_epoch

In [ ]:
no_improve = 0
best_val_loss = float("inf")
best_model_path = "best_g2p_model.pt"

train_losses = []
val_losses = []

print(f"\nStarting training for {NUM_EPOCHS} epochs...\n")

for epoch in range(NUM_EPOCHS):

    train_loss = train_epoch(
        model=model,
        dataloader=train_loader,
        criterion=CRITERION,
        optimizer=OPTIMIZER,
        scheduler=SCHEDULER,
        scaler=SCALER,
        device=DEVICE,
    )

    val_loss = evaluate(model, val_loader, CRITERION, DEVICE)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve = 0
        torch.save(model.state_dict(), best_model_path)
        status = "(Best)"
    else:
        no_improve += 1
        status = f"(No improvement for {no_improve} epochs)"

    if (epoch + 1) % 5 == 0:
        print(
            f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} {status}"
        )

    if no_improve >= PATIENCE:
        print(f"\nStopping early (patience={PATIENCE})")
        print(f"Best validation loss: {best_val_loss:.4f}")
        print("Restoring best model...")

        model.load_state_dict(
            torch.load(best_model_path, map_location=DEVICE)
        )
        break


Starting training for 100 epochs...

Epoch   1/100 | Train Loss: 3.0977 | Val Loss: 2.7578 (New Best! Model saved)
Epoch   2/100 | Train Loss: 2.5789 | Val Loss: 2.2556 (New Best! Model saved)
Epoch   3/100 | Train Loss: 2.0820 | Val Loss: 1.9170 (New Best! Model saved)
Epoch   4/100 | Train Loss: 1.8130 | Val Loss: 1.7886 (New Best! Model saved)
Epoch   5/100 | Train Loss: 1.6578 | Val Loss: 1.7510 (New Best! Model saved)
Epoch   6/100 | Train Loss: 1.5571 | Val Loss: 1.5560 (New Best! Model saved)
Epoch   7/100 | Train Loss: 1.4827 | Val Loss: 1.4652 (New Best! Model saved)
Epoch   8/100 | Train Loss: 1.4223 | Val Loss: 1.3500 (New Best! Model saved)
Epoch   9/100 | Train Loss: 1.3765 | Val Loss: 1.3707 (No improvement for 1 epochs)
Epoch  10/100 | Train Loss: 1.3370 | Val Loss: 1.2864 (New Best! Model saved)
Epoch  11/100 | Train Loss: 1.3076 | Val Loss: 1.3633 (No improvement for 1 epochs)
Epoch  12/100 | Train Loss: 1.2741 | Val Loss: 1.2174 (New Best! Model saved)
Epoch  13/100 

## **Inference**

In [ ]:
import random
from src.architecture.inference import greedy_decode
from src.evaluation.metrics import compute_metrics

In [ ]:
test_pairs = test_loader.dataset.data_pairs
predictions = []
references = []

for i, (graphemes, true_phonemes) in enumerate(test_pairs):
    word = ''.join(graphemes)
    
    predicted_phonemes, _ = greedy_decode(model, word, grapheme_vocab, phoneme_vocab, DEVICE)
    
    predictions.append(predicted_phonemes)
    references.append(true_phonemes)

In [ ]:
per, wer = compute_metrics(predictions, references)
total_words = len(test_pairs)
print("\nMetrics:")
print(f"Total Words Tested: {total_words}")
print(f"Phoneme Error Rate (PER): {per:.2f}%")
print(f"Word Error Rate (WER):    {wer:.2f}%")

print("\nSample Predictions:")
sample_indices = random.sample(range(total_words), 5)
for idx in sample_indices:
    grap = ''.join(test_pairs[idx][0])
    print(f"\nWord:     {grap.upper()}")
    print(f"Expected: {' '.join(references[idx])}")
    print(f"Output:   {' '.join(predictions[idx])}")


Metrics:
Total Words Tested: 1824
Phoneme Error Rate (PER): 3.00%
Word Error Rate (WER):    15.84%

Sample Predictions:

Word:     CARVAJAL
Expected: k a ɾ v a h a l
Output:   k a ɾ b a h a l

Word:     ARAKIBOT
Expected: ʔ a ɾ a k i b o t
Output:   ʔ a ɾ a k i b o t

Word:     ORARYO
Expected: ʔ o ɾ a ɾ j o
Output:   ʔ o ɾ a ɾ j o

Word:     INANG
Expected: ʔ i n a ŋ
Output:   ʔ i n a ŋ

Word:     TIO
Expected: t j o
Output:   t i o ʔ o
